# Silver - Regiões Legadas

Normalização da tabela de mapeamento regional de estados (UF).

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'legado_regioes_pipe'
output_table_name = 'legado_regioes'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{output_table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{output_table_name}'

In [ ]:
from pyspark.sql.functions import col, trim, upper

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .select(
        upper(trim(col("estado"))).cast("string").alias("uf_estado"),
        trim(col("regiao")).cast("string").alias("nome_regiao")
    )
    .filter(col("uf_estado").isNotNull())
    .dropDuplicates(["uf_estado"])
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='full',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['nome_regiao'],
    chave_upsert='uf_estado'
)